# Workshop 4 — primer AE z32 normal-only en Tesla T4

Este notebook ejecuta **un único entrenamiento desde cero**, seguido de
calibración y evaluación de anomalías. Mantiene la arquitectura original y no
reanuda best8 ni su Adam. AE z64 y VAE β=0.5, 1 y 4 quedan pendientes.

Plan: ajustar normalidad solo con train; entrenar con TODOS sus normales;
seleccionar/early-stop con normales de un grupo de hablantes val; calibrar con
otro grupo; evaluar datos mixtos de un tercer grupo. Test tiene una celda
separada, desactivada, que exige modelo/scores/umbrales congelados.

Configuración inicial: batch 128, Adam lr=0.0003, 30 épocas como máximo, clipping
FP64 a5, control cada 512 updates y al final de época. Early stopping: cinco
evaluaciones sin mejora de 0.1 nats/secuencia tras al menos tres épocas completas.
Presupuesto4 h/150.000 updates, sin reinicios automáticos. Guardas numéricas y
deterioro activas desde el inicio; no se detiene solo por clipping frecuente.

La revisión local encontró entradas x idénticas entre train normal y val. El
pipeline las excluye de las vistas held-out ANTES de muestrear/puntuar; conserva
todos los normales de train y no modifica archivos originales. SHA256 detecta
coincidencia exacta, no semejanza semántica ni paráfrasis.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, zipfile
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")


In [ ]:
# @title Código nuevo y fuentes de solo lectura
ZIP_SOURCE = "Upload" # @param ["Upload", "Drive"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab_normal_only_ae.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_normal_only_ae" # @param {type:"string"}
DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
LOCAL_PROCESSED_DIR = "/content/workshop4_normal_only_data/processed" # @param {type:"string"}
OUTPUT_PARENT = "/content/drive/MyDrive/Workshop_4/normal_only_ae" # @param {type:"string"}
BEST8_BASELINE = "/content/drive/MyDrive/Workshop_4/runs/workshop4_clipping_v2_prueba12/ae_z32/checkpoints/best.pt" # @param {type:"string"}
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Subir exactamente el ZIP normal-only nuevo")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
with zipfile.ZipFile(zip_path) as archive:
    helpers = [n for n in archive.namelist() if n.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("ZIP de proyecto incompatible")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import extract_project_zip, stage_processed_parquet, runtime_info
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
assert (WORKSHOP_DIR/"src/train_normal_ae.py").is_file()
# Copiar únicamente train y val para acelerar E/S. Test queda en Drive sin leer.
PROCESSED_DIR = stage_processed_parquet(DRIVE_PROCESSED_DIR, LOCAL_PROCESSED_DIR)
print(runtime_info())
print("Baseline disponible:", Path(BEST8_BASELINE).is_file())


## Etiquetas, equilibrio y separación

Normal = y siguiente carácter modal de train. En los archivos locales es
espacio (ID1):562.253/2.997.217 ejemplos. El resto de clases es anomalía proxy,
no anomalía semántica. Las anomalías agregadas son aproximadamente81.2%; por eso
AP≈0.82 puede estar cerca del baseline de prevalencia. No cambiar dirección de
scores ni invertirlos por observar AUC<0.5.

Los 169 hablantes val se distribuyen en tres roles disjuntos por hash/seed 42.
Solo model_selection normal guía el mejor checkpoint y early stopping.
Calibration normal fija umbrales. Evaluation mixto no modifica el modelo ni
umbral. Se registra la prevalencia REAL del subconjunto puntuado.


In [ ]:
# @title Una ejecución, desde cero; revisar primero el plan
RUN_TRAINING = False # @param {type:"boolean"}
DEVICE = "cuda" # @param ["cuda", "cpu"]
TRAIN_DIR = Path(OUTPUT_PARENT) / f"ae_z32_normal_only_{run_id}"
training_config_path = WORKSHOP_DIR/"config/normal_only_ae.json"
training_config = json.loads(training_config_path.read_text())
display(training_config)
train_command = [sys.executable, "-m", "src.train_normal_ae",
                 "--config", str(training_config_path), "--processed-dir", str(PROCESSED_DIR),
                 "--device", DEVICE, "--output-dir", str(TRAIN_DIR)]
subprocess.run(train_command, cwd=WORKSHOP_DIR, check=True, timeout=30)
print("Salida nueva en Drive:", TRAIN_DIR)


In [ ]:
training_summary = None
if RUN_TRAINING:
    info = runtime_info()
    if DEVICE == "cuda" and not info["cuda_available"]:
        raise RuntimeError("Seleccionar runtime GPU T4 antes de entrenar")
    # Una sola invocación. El CLI supervisa el plazo y conserva checkpoints.
    completed = subprocess.run(train_command + ["--execute"], cwd=WORKSHOP_DIR,
                               check=False, timeout=training_config["max_seconds"] + 60)
    print("Código de salida:", completed.returncode)
    if (TRAIN_DIR/"summary.json").is_file():
        training_summary = json.loads((TRAIN_DIR/"summary.json").read_text())
        print(training_summary["status"], training_summary.get("reason"))
        print("Mejor exportado:", training_summary.get("best_inference"))
    else:
        print("Sin resumen final: ejecución parcial; revisar checkpoints y no relanzar automáticamente.")
else:
    print("Solo plan. No se ejecutaron actualizaciones de entrenamiento.")


In [ ]:
# Curvas del entrenamiento normal-only y utilidad de z, sin tocar calibración/test.
import pandas as pd
import matplotlib.pyplot as plt
if RUN_TRAINING and (TRAIN_DIR/"history.csv").is_file():
    history = pd.read_csv(TRAIN_DIR/"history.csv")
    display(history.tail())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    history.plot(x="global_updates", y=["train_reconstruction_nats_per_sequence",
                 "selection_reconstruction_nats_per_sequence"], ax=axes[0])
    history.plot(x="global_updates", y="swap_damage_nats_per_sequence", ax=axes[1])
    history.plot(x="global_updates", y="swap_probability_tv", ax=axes[2])
    for ax, title in zip(axes, ("CE normal (nats/secuencia)", "Daño por intercambiar z", "TV por intercambio")):
        ax.set_title(title)
        ax.grid(alpha=.2)
    plt.tight_layout()
    plt.show()


## Calibración y evaluación de desarrollo, con comparación pareada

La inferencia carga exclusivamente best_inference.pt del entrenamiento nuevo;
rechaza un checkpoint inicial sin updates, un smoke o una continuación de best8.
best8 solo se carga como baseline congelado opcional, sin restaurar Adam.

Se comparan sobre LAS MISMAS entradas nuevas: MSE categórica (score principal)
y NLL auxiliar. Cada modelo/score calibra su propio umbral con normales del
mismo grupo calibration. Los umbrales/modelos/score y hashes se sellan antes
del primer forward de evaluation. Scores altos implican anomalía; score igual
al umbral queda normal. alpha=.05 prefijado no garantiza FPR nominal bajo shift.

Se reportan ROC-AUC, PR-AUC trapezoidal y AP por separado, precision, recall,
F1, FPR, confusión y prevalencia. Los valores best8 comunicados sobre 2.021
ejemplos son contexto; la comparación formal vuelve a puntuar best8 sobre las
filas actuales. La metodología original de selección de best8 fue distinta.
En la comparación pareada también se excluyen de ambos modelos las entradas
que aparecieron en CUALQUIER clase de train, porque best8 vio todas las clases.


In [ ]:
# @title Evaluar el mejor normal-only, sin repetir entrenamiento
RUN_DEVELOPMENT_EVALUATION = True # @param {type:"boolean"}
COMPARE_BEST8 = True # @param {type:"boolean"}
# Si se reabre el runtime, indicar la carpeta de la ejecución normal-only anterior.
EXISTING_TRAINING_DIR = "" # @param {type:"string"}
candidate_dir = Path(EXISTING_TRAINING_DIR).expanduser() if EXISTING_TRAINING_DIR.strip() else TRAIN_DIR
candidate_summary = candidate_dir/"summary.json"
candidate = None
if candidate_summary.is_file():
    training_summary = json.loads(candidate_summary.read_text())
    frozen_best = training_summary.get("best_inference")
    if frozen_best:
        candidate = Path(frozen_best["path"])
EVAL_DIR = Path(OUTPUT_PARENT)/f"development_{run_id}"
evaluation_command = None
if candidate is not None and candidate.is_file():
    evaluation_command = [sys.executable, "-m", "src.evaluate_normal_ae", "--stage", "evaluate",
                          "--checkpoint", str(candidate), "--processed-dir", str(PROCESSED_DIR),
                          "--device", DEVICE, "--output-dir", str(EVAL_DIR)]
    if COMPARE_BEST8:
        if not Path(BEST8_BASELINE).is_file():
            raise FileNotFoundError("Falta best8 para comparación pareada; corregir ruta o desactivar COMPARE_BEST8")
        evaluation_command += ["--baseline-checkpoint", str(BEST8_BASELINE)]
    subprocess.run(evaluation_command, cwd=WORKSHOP_DIR, check=True, timeout=30)
    if RUN_DEVELOPMENT_EVALUATION:
        completed = subprocess.run(evaluation_command + ["--execute"], cwd=WORKSHOP_DIR,
                                   check=False, timeout=1860)
        print("Código de salida:", completed.returncode)
        if (EVAL_DIR/"summary.json").is_file():
            evaluation_summary = json.loads((EVAL_DIR/"summary.json").read_text())
            print(evaluation_summary["status"], evaluation_summary.get("reason"))
            print("Test ready:", evaluation_summary.get("test_ready"))
else:
    print("No hay mejor normal-only entrenado exportado. Entrenar primero o indicar EXISTING_TRAINING_DIR.")


In [ ]:
def show_detection_metrics(folder):
    path = Path(folder)/"metrics.json"
    if not path.is_file():
        return
    payload = json.loads(path.read_text())
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for model, scores in payload["models"].items():
        for score, m in scores.items():
            rows.append({"modelo": model, "score": score, "n": m["examples"],
                         "prevalencia anomalía": m["anomaly_prevalence"], "ROC-AUC": m["roc_auc"],
                         "PR-AUC trapecio": m["pr_auc_trapezoidal"], "AP": m["average_precision"],
                         "precision": m["precision"], "recall": m["recall"], "F1": m["f1"],
                         "FPR": m["false_positive_rate"], "confusión": m["confusion"]})
            roc, pr = m["roc_curve"], m["precision_recall_curve"]
            if roc:
                axes[0].plot([p["fpr"] for p in roc], [p["tpr"] for p in roc], label=f"{model}/{score}")
            if pr:
                axes[1].plot([0]+[p["recall"] for p in pr], [1]+[p["precision"] for p in pr], label=f"{model}/{score}")
    display(pd.DataFrame(rows))
    axes[0].plot([0, 1], [0, 1], "k--", lw=.7)
    axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="PR; AP se calcula por grupos de empates")
    for ax in axes:
        ax.grid(alpha=.2)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
show_detection_metrics(EVAL_DIR)


## Test final: separado y desactivado

Activar únicamente después de fijar modelo, scores y umbrales. Se exige el
artefacto frozen_protocol.json y su SHA. Se verifica que no hayan cambiado
modelo, normalidad, roles, contenido de train o configuración. No se relee val
para recalibrar: solo train para comprobar hashes y test para puntuar.

El best8 histórico continúa siendo exploratorio. Los resultados de desarrollo
ayudan a entender diferencias; test se usa una vez para el protocolo fijado.
Una ejecución detenida o sin mejor candidato entrenado requiere revisar el
estado antes de declarar resultados finales. No se abren AE z64 ni VAE aquí.


In [ ]:
RUN_FINAL_TEST = False # @param {type:"boolean"}
FROZEN_PROTOCOL = str(EVAL_DIR/"frozen_protocol.json") # @param {type:"string"}
TEST_DIR = Path(OUTPUT_PARENT)/f"final_test_{run_id}"
if RUN_FINAL_TEST:
    protocol_path = Path(FROZEN_PROTOCOL)
    if candidate is None or not protocol_path.is_file() or not protocol_path.with_suffix(".sha256").is_file():
        raise FileNotFoundError("Faltan candidato/protocolo congelado y SHA")
    # Test sigue en Drive; no se copió ni leyó durante el entrenamiento.
    test_command = [sys.executable, "-m", "src.evaluate_normal_ae", "--stage", "test",
                    "--checkpoint", str(candidate), "--frozen-protocol", str(protocol_path),
                    "--processed-dir", str(DRIVE_PROCESSED_DIR), "--device", DEVICE,
                    "--output-dir", str(TEST_DIR)]
    subprocess.run(test_command, cwd=WORKSHOP_DIR, check=True, timeout=30)
    subprocess.run(test_command + ["--execute"], cwd=WORKSHOP_DIR, check=True, timeout=1860)
    show_detection_metrics(TEST_DIR)
else:
    print("Test permanece sin evaluar.")


Conservar en Drive history.csv, updates.jsonl, checkpoints, mejor exportado,
definición normal, partición y guardas de contenido; después scores.csv,
metrics.json, comparison.json y frozen_protocol.json con su SHA. No se
sobrescriben modelos históricos. Este primer experimento puede terminar antes
de 30 épocas por early stopping o guardas; reportar épocas/updates realmente
completados y alcance muestral. No se garantiza que normal-only resuelva una
definición proxy de anomalía; el objetivo es medirlo con un protocolo limpio.
